# 01 — Text Generation with Autoregressive (GPT-style) Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand the **autoregressive principle** behind GPT models: predict the next token, then feed it back in
- **Build and train a character-level language model** that generates text with exactly that mechanism
- Control generation with **temperature** and seed text
- Know how this scales up to GPT-2/GPT-4 (and why we do not download one here)

## 🔗 Prerequisites

- ✅ RNN/LSTM sequence models (Course 07/08)
- ✅ PyTorch training loops (Course 08)

---

This notebook covers practical activities from **Course 10, Unit 2**:
- Implementing text generation with GPT-style autoregressive models

---

## Introduction

**GPT models** are autoregressive language models: they model p(next token | all previous tokens) and generate by sampling one token at a time. This notebook demonstrates the same principle with a character-level LSTM — identical generation mechanics, no multi-GB download. Fine-tuning pretrained models is the subject of example 02; quality metrics are example 06.


## How Autoregressive Generation Works

Training: slice text into (context → next character) pairs and minimize cross-entropy on the next character. Generation: start from a seed, predict a distribution over the next character, **sample** from it (temperature scales the logits first: low = safe/repetitive, high = diverse/risky), append, repeat. Every LLM you have used works this way, token by token.


In [1]:
# WHAT/WHY: name the real GPT models this lesson maps onto, and state exactly
# what our stand-in shares with them (the autoregressive mechanism).
import torch, torch.nn as nn, torch.optim as optim, numpy as np
print(f'PyTorch {torch.__version__}')
print('Text Generation with Autoregressive Language Models')
print('=' * 60)
print('GPT models (concept):')
print('  - GPT-2: 117M-1.5B parameter transformer trained on web text')
print('  - Core idea: predict next token, left-to-right (causal LM)')
print()
print('This notebook demonstrates the SAME autoregressive principle')
print('using a character-level LSTM - identical inference mechanics,')
print('no multi-GB download required.')


PyTorch 2.13.0
Text Generation with Autoregressive Language Models
GPT models (concept):
  - GPT-2: 117M-1.5B parameter transformer trained on web text
  - Core idea: predict next token, left-to-right (causal LM)

This notebook demonstrates the SAME autoregressive principle
using a character-level LSTM - identical inference mechanics,
no multi-GB download required.


In [2]:
# WHAT/WHY: build the full pipeline on a tiny corpus first — encode text,
# train next-character prediction, then generate by sampling in a loop.
# ── Character-level language model (same autoregressive principle as GPT) ──
torch.manual_seed(42)
corpus = ('the cat sat on the mat. the cat ate a rat. ' * 8 +
          'once upon a time a fox jumped over the fence. ' * 8)
# ── Vocabulary: map each character to an integer id (a tiny "tokenizer") ──
chars = sorted(set(corpus))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
V, S = len(chars), 30


# ── Dataset: every 30-char window predicts the character that follows it ──
X = torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)], dtype=torch.long)
y = torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)], dtype=torch.long)

# ── Model: embedding → LSTM → linear scores over the vocabulary ───────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, 32)
        self.lstm = nn.LSTM(32, 128, batch_first=True)
        self.fc   = nn.Linear(128, V)
    def forward(self, x):
        return self.fc(self.lstm(self.emb(x))[0][:, -1, :])

model = CharLM()
opt   = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)

# ── Training: minimize next-character cross-entropy for 15 epochs ─────────
for epoch in range(15):
    model.train(); el = 0
    for xb, yb in loader:
        opt.zero_grad(); loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step(); el += loss.item()
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}: loss={el/len(loader):.4f}')

# ── Greedy sampling (like GPT greedy decoding) ────────────────────────────
def generate(seed, n=60, temperature=0.8):
    model.eval(); out = seed
    ctx = [c2i.get(c, 0) for c in seed[-S:]]
    for _ in range(n):
        x = torch.tensor([ctx], dtype=torch.long)
        with torch.no_grad():
            logits = model(x)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        nxt = int(np.random.choice(V, p=probs))
        out += i2c[nxt]; ctx = ctx[1:] + [nxt]
    return out

print('\nSeed → generated:')
print(repr(generate('the cat ', n=60, temperature=0.8)))
print(repr(generate('once upon', n=60, temperature=1.0)))

Epoch 5: loss=1.6068


Epoch 10: loss=0.4866


Epoch 15: loss=0.1236

Seed → generated:
'the cat sat on the cat ate a rat. the cat at on the mat on the cat s'
'once upon a time a fox jumped over the fence. on the a time a  aox ju'


## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [3]:
# WHAT/WHY: the same pipeline on a richer corpus (Shakespeare) with a
# 2-layer LSTM — watch the loss fall and then read the generated text.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

# ── Vocabulary: map each character to an integer id ───────────────────────
chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]


# ── Dataset: every 20-char window predicts the next character ─────────────
SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

# ── Training: 200 quick steps, a fresh random 256-window batch each time ──
for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

Epoch 0 — loss: 3.169


Epoch 50 — loss: 1.298


Epoch 100 — loss: 0.053


Epoch 150 — loss: 0.011



── Generated Text ──────────────────────────────────────────────
to be or notek a sa ehat akse that is the tis ta cobues arin to sleep no more and by a sleep to say we end the h

This is exactly how ChatGPT generates text — one token at a time.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

You built **autoregressive text generation** — the core mechanism behind GPT, ChatGPT, and LLaMA. The model learns to predict the next token given context. Temperature controls creativity vs coherence. Scaling this architecture to billions of parameters creates foundation models.